In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import cftime
import dask
import matplotlib.pyplot as plt
import os
import xesmf as xe
import climpred
import ocetrac as ot
import pop_tools

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.util import add_cyclic_point

In [46]:
rad_val = 3
fold = '/glade/work/jtcohen'

In [53]:
# # IF JUST PREPROCESSING OBSERVATIONS, JUST RUN THIS CELL

# for radius_val in [1, 2, 3, 4, 5, 6, 7]:
#     long_ver = xr.open_dataset(f'{fold}/OISST_features_premask.r{radius_val}.TEMP.1989-2020.nc')
#     obs = long_ver['TEMP'].where(long_ver['features']>0).drop_vars(['month', 'quantile'])
#     obs['time'] = [cftime.DatetimeNoLeap(y, m, 1, 0, 0, 0, 0, has_year_zero=True) for y, m in zip(obs['time.year'], obs['time.month'])]
#     obs.attrs['Conventions'] = 'CF-1.8'
#     obs.attrs['long_name'] = 'temperature'
#     obs.attrs['units'] = 'degrees_Celsius'
#     obs.attrs['standard_name'] = 'sea_surface_temperature'
#     obs.attrs['ocetrac_radius'] = f'{rad_val}'
#     obs['lat'].attrs['long_name'] = 'latitude'
#     obs['lat'].attrs['units'] = 'degrees_north'
#     obs['lat'].attrs['standard_name'] = 'latitude'
#     obs['lon'].attrs['long_name'] = 'longitude'
#     obs['lon'].attrs['units'] = 'degrees_east'
#     obs['lon'].attrs['standard_name'] = 'longitude'
#     obs['time'].attrs['long_name'] = 'time'
#     obs['time'].attrs['standard_name'] = 'time'
#     obs.to_netcdf(f'{fold}/MODE_files_final/observation_r{radius_val}.nc')

In [27]:
def setup_axes(ax):
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=0)
    ax.coastlines(resolution='50m', color='black', lw=0.5)
    ax.set_ylabel('latitude')
    ax.set_xlabel('longitude')


def lons_to_360(data, coord='lon'):
    """ Converts longitude coordinates from (-180, 180) to (0, 360)."""
    data.coords[coord] = (360 + (data.coords[coord] % 360)) % 360
    data = data.sortby(data[coord])
    return data


def regrid_SMYLE(ds, glat=1, glon=1):
    """
    Inputs:
        ds: xr.DataArray with coordinates that include TLAT and TLONG
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds = ds.rename(({'TLONG': 'lon', 'TLAT': 'lat'}))
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

In [28]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="40GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=40GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="02:00:00",  # Amount of wall time
    interface="ext",  # Interface to use
)

# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=2) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

# show the client that you have been assigned, you can click on the link and it will show you 
# a dashboard with all the tasks that have to be performed to do your calculation
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.63:38001,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Load Observations

In [29]:
obs = xr.open_dataset(f'{fold}/OISST_features_premask.r{rad_val}.TEMP.1989-2020.nc').drop_vars(['quantile', 'month'])
obs['time'] = [cftime.DatetimeNoLeap(y, m, 1, 0, 0, 0, 0, has_year_zero=True) for y, m in zip(obs['time.year'], obs['time.month'])]

In [6]:
# fosiobs = xr.open_dataset(f'{fold}/FOSI_features_premask.r{rad_val}.TEMP.1989-2020.nc').drop_vars(['quantile', 'month'])
# fosiobs['time'] = [cftime.DatetimeNoLeap(y, m, 1, 0, 0, 0, 0, has_year_zero=True) for y, m in zip(fosiobs['time.year'], fosiobs['time.month'])]

# Load Forecasts

In [13]:
firstyear = 1989
lastyear = 2018
startmonths = [2, 5, 8, 11]
radius_val = rad_val
field = 'TEMP'

files = [f'SMYLE_features_premask.r{radius_val}.{field}.{startmonth:02d}.{firstyear}-{lastyear}.nc' for startmonth in startmonths]

dss = [xr.open_dataset(f'{fold}/{file}') for file in files]

In [14]:
%%time
features = {}
for i, ds in enumerate(dss):
    sm = startmonths[i]
    features[sm] = ds.drop_vars(['z_t', 'quantile']).rename({'Y': 'init', 'M': 'member', 'L': 'lead'})
    features[sm]['lead'] = features[sm]['lead'] - 1 # lead 1 is really lead 1/2
    features[sm]['init'] = xr.cftime_range(start=f'{firstyear}-{sm:02d}-01',
                                  freq='12MS',
                                  periods=features[sm].init.size,
                                  calendar='noleap')
    features[sm]['lead'].attrs['units'] = 'months'

CPU times: user 6.41 ms, sys: 3.41 ms, total: 9.82 ms
Wall time: 10 ms


In [15]:
%%time
smyle_sorted = xr.concat(list(features.values()), dim='init').sortby('init')

CPU times: user 40 s, sys: 56.9 s, total: 1min 36s
Wall time: 4min


In [16]:
smyle_sorted['features'] = xr.where(~np.isnan(smyle_sorted['features']), 1, 0)

In [17]:
smyle = smyle_sorted.transpose('init', 'lead', 'member', 'lat', 'lon')

# Climpred Hindcast

In [18]:
hindcast = climpred.HindcastEnsemble(smyle)
hindcast = hindcast.add_observations(obs)

In [19]:
fcst_ds = hindcast.get_initialized()
obs_ds = hindcast.get_observations()

In [20]:
obs_mask = xr.where(~np.isnan(obs_ds['TEMP'].mean('time')), 1, 0)
fcst_mask = xr.where(~np.isnan(fcst_ds['TEMP'].isel(init=0, lead=0, member=0)), 1, 0)
fullmask = xr.where((obs_mask==0) | (fcst_mask==0), 0, 1)
# new_fcst_mask = xr.where(~np.isnan(fcst_ds['TEMP'].isel(init=0, lead=0, member=0).where(fullmask==1, np.nan)), 1, 0)

In [21]:
fcst = fcst_ds['TEMP'].where(fcst_ds['features']>0).where(fullmask==1, np.nan)
obs = obs_ds['TEMP'].where(obs_ds['features']>0)

In [22]:
fcst.attrs['Conventions'] = 'CF-1.8'
fcst.attrs['long_name'] = 'temperature'
fcst.attrs['units'] = 'degrees_Celsius'
fcst.attrs['standard_name'] = 'sea_surface_temperature'
fcst.attrs['ocetrac_radius'] = f'{rad_val}'
fcst['lat'].attrs['long_name'] = 'latitude'
fcst['lat'].attrs['units'] = 'degrees_north'
fcst['lat'].attrs['standard_name'] = 'latitude'
fcst['lon'].attrs['long_name'] = 'longitude'
fcst['lon'].attrs['units'] = 'degrees_east'
fcst['lon'].attrs['standard_name'] = 'longitude'

obs.attrs['Conventions'] = 'CF-1.8'
obs.attrs['long_name'] = 'temperature'
obs.attrs['units'] = 'degrees_Celsius'
obs.attrs['standard_name'] = 'sea_surface_temperature'
obs.attrs['ocetrac_radius'] = f'{rad_val}'
obs['lat'].attrs['long_name'] = 'latitude'
obs['lat'].attrs['units'] = 'degrees_north'
obs['lat'].attrs['standard_name'] = 'latitude'
obs['lon'].attrs['long_name'] = 'longitude'
obs['lon'].attrs['units'] = 'degrees_east'
obs['lon'].attrs['standard_name'] = 'longitude'
obs['time'].attrs['long_name'] = 'time'
obs['time'].attrs['standard_name'] = 'time'

# Save

In [23]:
fcst.load().to_netcdf(f'{fold}/MODE_files_final/forecast_r{rad_val}.nc')
obs.load().to_netcdf(f'{fold}/MODE_files_final/observation_r{rad_val}.nc')

In [24]:
xr.open_dataset(f'{fold}/MODE_files_final/forecast_r5.nc')['TEMP']

<xarray.DataArray 'TEMP' (init: 120, lead: 24, member: 20, lat: 180, lon: 360)>
[3732480000 values with dtype=float64]
Coordinates:
  * member   (member) int64 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
  * lead     (lead) int64 0 1 2 3 4 5 6 7 8 9 ... 14 15 16 17 18 19 20 21 22 23
  * init     (init) object 1989-02-01 00:00:00 ... 2018-11-01 00:00:00
  * lat      (lat) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 -179.5 -178.5 -177.5 -176.5 ... 177.5 178.5 179.5
Attributes:
    regrid_method:   bilinear
    Conventions:     CF-1.8
    long_name:       temperature
    units:           degrees_Celsius
    standard_name:   sea_surface_temperature
    ocetrac_radius:  5